# 05 — Indicator Exploration

Computes and visualises thermal indicators across spatial units.
- Loads spatial units and annotation data
- Computes annotation-based indicators (Scenario C — always available)
- Visualises priority index choropleth
- Exports tables for article

In [ ]:
from __future__ import annotations

import pathlib
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from gdynia_thermal_audit.settings import Settings
from gdynia_thermal_audit.logging_utils import setup_logging
from gdynia_thermal_audit.indicators.annotation_indicators import compute_annotation_indicators
from gdynia_thermal_audit.indicators.priority_index import compute_priority_index
from gdynia_thermal_audit.spatial_units.districts import get_gdynia_districts_placeholder
from gdynia_thermal_audit.reporting.figures import plot_indicators_map
from gdynia_thermal_audit.utils.io import ensure_dir

setup_logging()
settings = Settings()
ensure_dir('outputs/figures')
ensure_dir('outputs/tables')

In [ ]:
# Load spatial units
try:
    gdf_districts = gpd.read_file('data/processed/spatial_units.gpkg', layer='districts')
except Exception:
    gdf_districts = get_gdynia_districts_placeholder()
    print('Using placeholder districts')

# Load annotations
try:
    df_ann = pd.read_csv('data/annotations/annotations.csv')
except FileNotFoundError:
    df_ann = pd.read_csv('data/demo/demo_annotations.csv')
    print('Using demo annotations')

print(f'Districts: {len(gdf_districts)}, Annotations: {len(df_ann)}')

In [ ]:
# Compute annotation-based indicators
df_indicators = compute_annotation_indicators(gdf_districts, df_ann)
print(df_indicators.columns.tolist())
df_indicators.head()

In [ ]:
# Compute priority index
weights = {'anomaly_count': 0.4, 'mean_intensity': 0.3, 'building_anomaly_share': 0.3}
df_indicators = compute_priority_index(df_indicators, weights)
df_indicators[['unit_id', 'anomaly_count', 'priority_index']].sort_values('priority_index', ascending=False)

In [ ]:
# Merge indicators onto spatial units and plot
gdf_merged = gdf_districts.merge(df_indicators, left_on='district_id', right_on='unit_id', how='left')
if 'priority_index' in gdf_merged.columns:
    plot_indicators_map(gdf_merged, df_indicators, column='priority_index',
                        output_path='outputs/figures/priority_index.png')
else:
    print('priority_index not available; check indicator computation')